# 模型对比分析报告

本 Notebook 汇总三个 CNN 模型（MiniCNN / VGGLite / MicroResNet）的训练与评估结果，进行系统性对比。

- 训练历史来源：优先读取 `training/runs/<模型>/<最新 run>/history.json`（新格式，配置可追溯）；
  无新 run 时回退读取 `training/logs/<模型>/history.json`（legacy 旧日志，信息不全，会明确标注）。
- 测试指标来源：统一评估入口 `utils/evaluation.evaluate_checkpoint`（显式 checkpoint + split；
  权重 SHA-256、逐样本预测与概率、划分哈希随结果保存到 `analysis/evaluations/`，指标可重算）。


In [ ]:
import sys

sys.path.insert(0, '..')

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

from training.trainer import PROJECT_ROOT

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

config_path = PROJECT_ROOT / 'configs' / 'training_config.yaml'
with open(config_path, encoding='utf-8') as f:
    config = yaml.safe_load(f)

CLASS_NAMES = config['data']['class_names']
MODEL_NAMES = ['mini_cnn', 'vgg_lite', 'micro_resnet']


def find_latest_run(model_name):
    """返回 (history 路径, 来源标签)；优先新格式 runs，回退 legacy logs。"""
    runs_dir = PROJECT_ROOT / 'training' / 'runs' / model_name
    if runs_dir.exists():
        run_dirs = sorted(
            (p for p in runs_dir.iterdir() if p.is_dir()),
            key=lambda p: p.stat().st_mtime, reverse=True,
        )
        for run_dir in run_dirs:
            hist = run_dir / 'history.json'
            if hist.exists():
                return hist, f'runs/{run_dir.name}'
    legacy = PROJECT_ROOT / 'training' / 'logs' / model_name / 'history.json'
    if legacy.exists():
        return legacy, 'legacy(旧日志·配置信息不全)'
    return None, None


histories, history_sources = {}, {}
for name in MODEL_NAMES:
    path, label = find_latest_run(name)
    if path is not None:
        with open(path, encoding='utf-8') as f:
            histories[name] = json.load(f)
        history_sources[name] = label
        n = len(histories[name]['train_loss'])
        print(f"已加载 {name} 训练历史 ({n} epochs) | 来源: {label}")
    else:
        print(f"警告: {name} 无可用训练历史（runs/ 与 legacy logs 均为空）")


## 1. 训练曲线对比

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = {'mini_cnn': 'blue', 'vgg_lite': 'green', 'micro_resnet': 'red'}

for name, hist in histories.items():
    epochs = range(1, len(hist['train_loss']) + 1)
    c = colors[name]
    axes[0, 0].plot(epochs, hist['train_loss'], color=c, label=f'{name}', linestyle='-')
    axes[0, 1].plot(epochs, hist['val_loss'], color=c, label=f'{name}', linestyle='-')
    axes[1, 0].plot(epochs, hist['train_acc'], color=c, label=f'{name}', linestyle='-')
    axes[1, 1].plot(epochs, hist['val_acc'], color=c, label=f'{name}', linestyle='-')

axes[0, 0].set_title('Train Loss')
axes[0, 0].legend()
axes[0, 1].set_title('Val Loss')
axes[0, 1].legend()
axes[1, 0].set_title('Train Accuracy')
axes[1, 0].legend()
axes[1, 1].set_title('Val Accuracy')
axes[1, 1].legend()

for ax in axes.flat:
    ax.set_xlabel('Epoch')

plt.suptitle('三模型训练曲线对比（来源见加载日志；legacy 日志为信息不全的旧运行）', fontsize=12)
plt.tight_layout()
curve_dir = PROJECT_ROOT / 'analysis' / 'training_curves'
curve_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(curve_dir / 'comparison_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 2. 最优性能汇总（参数量动态计算）


In [ ]:
# 汇总表：参数量由 model_spec 动态计算；Val 指标来自各自训练历史（来源见最后一列）
from utils.model_spec import build_model_from_spec, count_parameters, make_spec_from_config

summary = []
for name in MODEL_NAMES:
    if name not in histories:
        continue
    hist = histories[name]
    best_epoch = int(np.argmax(hist['val_acc'])) + 1
    spec = make_spec_from_config(config, name)
    total, _ = count_parameters(build_model_from_spec(spec, allow_unknown_dropout=True))
    summary.append({
        '模型': name,
        '参数量': f'{total:,}',
        '最优Epoch': best_epoch,
        '最优Val Acc': f"{hist['val_acc'][best_epoch - 1]:.4f}",
        '最优Val Loss': f"{hist['val_loss'][best_epoch - 1]:.4f}",
        '最终Train Acc': f"{hist['train_acc'][-1]:.4f}",
        '总Epochs': len(hist['train_loss']),
        '历史来源': history_sources[name],
    })

summary_df = pd.DataFrame(summary)
print("=== 模型性能汇总（Val=验证集口径，与测试指标分开列示）===")
summary_df


## 3. 混淆矩阵对比

In [ ]:
# 加载保存的混淆矩阵图片
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
for i, name in enumerate(MODEL_NAMES):
    cm_path = PROJECT_ROOT / 'analysis' / 'confusion_matrices' / f'{name}_cm.png'
    if cm_path.exists():
        img = plt.imread(str(cm_path))
        axes[i].imshow(img)
        axes[i].axis('off')
        axes[i].set_title(name)
    else:
        axes[i].text(0.5, 0.5, f'{name}\n(未生成)', ha='center', va='center', fontsize=14)
        axes[i].axis('off')

plt.suptitle('混淆矩阵对比', fontsize=14)
plt.tight_layout()
out = PROJECT_ROOT / 'analysis' / 'comparison_confusion_matrices.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()


## 4. 各类别 Recall 对比（统一评估入口）


In [ ]:
# 统一评估入口：对推理目录中的权重评估 PrivateTest（显式 checkpoint，来源可追溯）
# 评估产物保存到 analysis/evaluations/（含权重 SHA-256、逐样本预测与概率，可复算 ECE/NLL）
from sklearn.metrics import classification_report

from utils.evaluation import evaluate_checkpoint

SPLIT = 'PrivateTest'

eval_results = {}
for name in MODEL_NAMES:
    ckpt_path = PROJECT_ROOT / 'inference' / 'saved_models' / f'{name}_best.pth'
    if not ckpt_path.exists():
        print(f'跳过 {name}: 权重不存在 ({ckpt_path})')
        continue
    print(f'--- {name} ---')
    eval_results[name] = evaluate_checkpoint(ckpt_path, split=SPLIT, device='cpu')

# 指标汇总（权重 SHA-256 前 16 位用于核对来源）
rows = []
for name, r in eval_results.items():
    m = r['metrics']
    rows.append({
        '模型': name,
        'accuracy': f"{m['accuracy']:.4f}",
        'macro_f1': f"{m['macro_f1']:.4f}",
        'balanced_acc': f"{m['balanced_accuracy']:.4f}",
        'ECE(15bin)': f"{m['ece']:.4f}",
        '权重SHA256(前16)': r['checkpoint']['sha256'][:16],
    })
print(f"\n=== {SPLIT} 指标汇总（统一评估入口）===")
display(pd.DataFrame(rows))

# 各类别 recall 对比图
if eval_results:
    x = np.arange(len(CLASS_NAMES))
    width = 0.25
    model_colors = {'mini_cnn': '#3498db', 'vgg_lite': '#2ecc71', 'micro_resnet': '#e74c3c'}

    fig, ax = plt.subplots(figsize=(12, 6))
    for i, (name, r) in enumerate(eval_results.items()):
        values = [r['metrics']['per_class'][cls]['recall'] for cls in CLASS_NAMES]
        bars = ax.bar(x + i * width, values, width, label=name,
                      color=model_colors.get(name), alpha=0.85)
        for bar, val in zip(bars, values, strict=True):
            ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.01,
                    f'{val:.2f}', ha='center', va='bottom', fontsize=8)

    ax.set_xlabel('情感类别')
    ax.set_ylabel('Recall')
    ax.set_title(f'各类别 Recall 对比（{SPLIT}，统一评估入口）')
    ax.set_xticks(x + width)
    ax.set_xticklabels(CLASS_NAMES)
    ax.set_ylim(0, 1.1)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    out = PROJECT_ROOT / 'analysis' / 'comparison_per_class_recall.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()

    first = eval_results[next(iter(eval_results))]
    support = first['metrics']['per_class']['Disgust']['support']
    print(f"注: {SPLIT} Disgust 支持数为 {support}；"
          f"一张样本约对应 {100 / support:.2f} 个百分点 recall 变化，解读时注意粒度")
else:
    print('无可用评估结果（saved_models 下无权重）')


## 5. 参数量与评估指标


In [ ]:
# 参数量 vs 准确率 散点图（参数量动态计算；不做“每千参数贡献”换算）
from utils.model_spec import build_model_from_spec, count_parameters, make_spec_from_config

if eval_results:
    ylabel = f'{SPLIT} accuracy（统一评估入口）'
else:
    ylabel = '最优验证准确率（尚未运行 §4 统一评估）'

fig, ax = plt.subplots(figsize=(8, 6))
for name in MODEL_NAMES:
    if name in eval_results:
        acc = eval_results[name]['metrics']['accuracy']
    elif name in histories:
        acc = max(histories[name]['val_acc'])
        print(f"注: {name} 使用训练历史的 val_acc 作为纵坐标（未纳入统一评估）")
    else:
        continue
    spec = make_spec_from_config(config, name)
    total, _ = count_parameters(build_model_from_spec(spec, allow_unknown_dropout=True))
    ax.scatter(total, acc, s=200, zorder=5)
    ax.annotate(name, (total, acc), textcoords='offset points', xytext=(10, 10), fontsize=12)

ax.set_xlabel('参数量')
ax.set_ylabel(ylabel)
ax.set_title('参数量 vs 准确率')
ax.set_xscale('log')
plt.tight_layout()
out = PROJECT_ROOT / 'analysis' / 'comparison_param_efficiency.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()


## 6. 结论与模型选择

根据以上对比分析记录结论：
- 准确率最优模型与参数效率最优模型是否一致（见 §2 汇总表与 §5 散点图）
- 各模型在哪些类别上差异显著（见 §4 各类别 Recall；同时参考 Macro-F1 与 balanced accuracy）
- 部署到推理应用：使用 `tools/export_model.py` 显式导出所选 run 的 best.pth，
  导出清单会记录来源 run 与 SHA-256；本 Notebook 不自动复制任何文件


In [ ]:
# 结论汇总：不自动复制文件；导出请使用 tools/export_model.py（避免意外更换推理目录权重）
if eval_results:
    best_name = max(eval_results, key=lambda n: eval_results[n]['metrics']['accuracy'])
    m = eval_results[best_name]['metrics']
    print(f"评估口径: {SPLIT}（统一评估入口）")
    print(f"  accuracy 最优: {best_name} ({m['accuracy']:.4f})")
    print("注意: accuracy 与 Macro-F1 排名可能不一致，请联合讨论；")
    print("      不同模型评估所用权重均为可追溯的显式文件（SHA-256 见 §4 汇总表）")
    print()
    print("导出命令示例:")
    print("  python tools/export_model.py --list  # 查看已有导出与来源")
    print(f"  python tools/export_model.py --checkpoint "
          f"training/runs/{best_name}/<run_id>/checkpoints/best.pth")
elif histories:
    print("尚未运行统一评估（§4），无法给出评估口径结论")
else:
    print("无可用数据")
